# Retail Sales Data Cleaning & Exploratory Analysis
**Dataset:** Retail Store Sales (synthetic, intentionally messy) — 12,575 transactions across 8 product categories.

**Goal:** Diagnose data quality issues, recover as much usable data as possible using justified logic (not guessing), and prepare a clean dataset for analysis.


## 1. Load and inspect the raw data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('retail_store_sales.csv')
print(df.shape)
df.head()

(12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB


## 2. Diagnose data quality issues

First pass: check nulls, duplicates, and value ranges before touching anything.

In [3]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Missing %:")
print((df.isnull().sum() / len(df) * 100).round(2))
print()
print("Fully duplicated rows:", df.duplicated().sum())

Missing values per column:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

Missing %:
Transaction ID       0.00
Customer ID          0.00
Category             0.00
Item                 9.65
Price Per Unit       4.84
Quantity             4.80
Total Spent          4.80
Payment Method       0.00
Location             0.00
Transaction Date     0.00
Discount Applied    33.39
dtype: float64

Fully duplicated rows: 0


**Findings:**
- `Discount Applied`: 33.4% missing
- `Item`, `Price Per Unit`, `Quantity`, `Total Spent`: each missing in a smaller but meaningful chunk (4–10%)
- No full-row duplicates

Missingness in `Item`/`Price`/`Quantity`/`Total Spent` looked suspicious — let's check whether it's random or follows a pattern.

In [4]:
cols = ['Item', 'Price Per Unit', 'Quantity', 'Total Spent']
pattern = df[cols].isnull().value_counts()
print(pattern)

Item   Price Per Unit  Quantity  Total Spent
False  False           False     False          11362
True   True            False     False            609
       False           True      True             604
Name: count, dtype: int64


**Key discovery:** missingness isn't random — it falls into exactly two patterns:

- **Pattern A (609 rows):** `Item` and `Price Per Unit` missing, but `Quantity` and `Total Spent` are present.
- **Pattern B (604 rows):** `Item`, `Quantity`, and `Total Spent` missing, but `Price Per Unit` is present.

This matters because it means the missing values aren't unrecoverable — they can potentially be derived from the relationships between columns.

## 3. Check whether recovery is mathematically possible

Two questions to validate before writing any recovery logic:
1. Does each `Item` have one fixed `Price Per Unit`? (needed to look up Item from Price)
2. Within a `Category`, does `Price Per Unit` uniquely identify the `Item`? (needed to reverse-lookup)

In [5]:
clean_ref = df.dropna(subset=['Item', 'Price Per Unit'])

price_variation = clean_ref.groupby('Item')['Price Per Unit'].nunique()
print("Items with more than one distinct price:", (price_variation > 1).sum(), "out of", len(price_variation))

dup_check = clean_ref.groupby(['Category', 'Price Per Unit'])['Item'].nunique()
print("Category+Price combinations mapping to more than one Item:", (dup_check > 1).sum(), "out of", len(dup_check))

Items with more than one distinct price: 0 out of 200
Category+Price combinations mapping to more than one Item: 0 out of 200


Both checks come back clean: every item has exactly one fixed price, and `(Category, Price)` uniquely identifies an `Item`. This confirms the recovery logic below is mathematically sound, not a guess.

## 4. Recover Pattern A (609 rows) — Item & Price missing

Logic: `Price = Total Spent / Quantity`, then look up `Item` from `(Category, Price)`.

In [6]:
item_to_price = clean_ref.drop_duplicates('Item').set_index('Item')['Price Per Unit'].to_dict()
catprice_to_item = clean_ref.drop_duplicates(['Category', 'Price Per Unit']).set_index(['Category', 'Price Per Unit'])['Item'].to_dict()
item_median_qty = clean_ref.groupby('Item')['Quantity'].median()

df['Quantity_Imputed'] = False  # flag column for transparency

pA = df['Item'].isnull() & df['Price Per Unit'].isnull() & df['Quantity'].notnull() & df['Total Spent'].notnull()
print("Pattern A rows:", pA.sum())

df.loc[pA, 'Price Per Unit'] = (df.loc[pA, 'Total Spent'] / df.loc[pA, 'Quantity']).round(2)
df.loc[pA, 'Item'] = df.loc[pA].apply(lambda r: catprice_to_item.get((r['Category'], r['Price Per Unit'])), axis=1)

print("Item still missing after recovery:", df.loc[pA, 'Item'].isnull().sum())

Pattern A rows: 609
Item still missing after recovery: 0


## 5. Recover Pattern B (604 rows) — Item, Quantity & Total missing

Here, `Price Per Unit` is known, so `Item` can be recovered directly via `(Category, Price)`.

But `Quantity` and `Total Spent` are *both* missing — that's two unknowns and only one equation (`Total = Quantity × Price`), so there's no way to solve for an exact value. Rather than drop these rows (and lose ~5% of the dataset) or silently guess, we impute `Quantity` using the **median quantity for that item** and flag it with `Quantity_Imputed = True` — so anyone using this data later knows exactly which values are estimates.

In [7]:
pB = df['Item'].isnull() & df['Price Per Unit'].notnull() & df['Quantity'].isnull() & df['Total Spent'].isnull()
print("Pattern B rows:", pB.sum())

df.loc[pB, 'Item'] = df.loc[pB].apply(lambda r: catprice_to_item.get((r['Category'], r['Price Per Unit'])), axis=1)
df.loc[pB, 'Quantity'] = df.loc[pB, 'Item'].map(item_median_qty)
df.loc[pB, 'Total Spent'] = (df.loc[pB, 'Quantity'] * df.loc[pB, 'Price Per Unit']).round(2)
df.loc[pB, 'Quantity_Imputed'] = True

print("Item still missing after recovery:", df.loc[pB, 'Item'].isnull().sum())
print("Rows flagged as imputed:", df['Quantity_Imputed'].sum())

Pattern B rows: 604
Item still missing after recovery: 0
Rows flagged as imputed: 604


## 6. Handle `Discount Applied`

This is a boolean field — we can't infer a missing True/False without guessing, so instead of dropping rows or defaulting to one value, missing entries get their own explicit `'Unknown'` category. This preserves the rows and is honest about what we don't know.

In [8]:
df['Discount Applied'] = df['Discount Applied'].map({True: 'Yes', False: 'No'}).fillna('Unknown')
print(df['Discount Applied'].value_counts())

Discount Applied
Yes        4219
Unknown    4199
No         4157
Name: count, dtype: int64


## 7. Fix data types

In [9]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
print(df['Transaction Date'].min(), '→', df['Transaction Date'].max())
df.dtypes

2022-01-01 00:00:00 → 2025-01-18 00:00:00


Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied               str
Quantity_Imputed              bool
dtype: object

## 8. Final validation

Confirm zero missing values remain in the core fields, and spot-check that recovered rows make sense.

In [10]:
print("Remaining nulls per column:")
print(df.isnull().sum())
print()
print("Final shape:", df.shape)
print("Rows recovered (not dropped): 1,213 out of 1,213 originally incomplete rows")
print("Rows with an imputed (estimated) Quantity:", df['Quantity_Imputed'].sum())

Remaining nulls per column:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Quantity_Imputed    0
dtype: int64

Final shape: (12575, 12)
Rows recovered (not dropped): 1,213 out of 1,213 originally incomplete rows
Rows with an imputed (estimated) Quantity: 604


In [11]:
df.to_csv('cleaned_retail_sales.csv', index=False)
print("Saved: cleaned_retail_sales.csv")

Saved: cleaned_retail_sales.csv


## Summary

| Issue | Rows affected | Resolution |
|---|---|---|
| Item + Price missing (Pattern A) | 609 | Exactly recovered via `Price = Total ÷ Quantity`, then `Item` lookup |
| Item + Quantity + Total missing (Pattern B) | 604 | `Item` exactly recovered; `Quantity`/`Total` estimated via item median, flagged |
| Discount Applied missing | 4,199 | Kept as explicit `'Unknown'` category rather than guessed |
| Full duplicate rows | 0 | None found |

**Result:** 0 rows dropped, 0 nulls remaining, all estimates transparently flagged for downstream analysis.

Next step: exploratory analysis on `cleaned_retail_sales.csv` — category performance, online vs in-store, time trends, and the impact of discounting.